In [4]:
import pandas as pd
import numpy as np
import scipy.io
from scipy.signal import hilbert, butter, filtfilt
import os
from tqdm import tqdm

# =============================================================================
# 1. 核心高级特征计算函数
# =============================================================================
def calculate_advanced_features(signal: np.ndarray, rpm: float, sampling_rate: int) -> dict:
    """
    为单条信号计算所有高级特征（边带+包络），
    并且同时基于DE和FE两种轴承的参数进行计算。
    """
    n_points = len(signal)
    adv_features = {}

    # --- 准备工作：计算所有理论频率 ---
    fr = rpm / 60.0
    bearings = {
        'DE': {'n': 9, 'd': 0.3126, 'D': 1.537},
        'FE': {'n': 9, 'd': 0.2656, 'D': 1.122}
    }
    
    theoretical_freqs = {}
    for name, p in bearings.items():
        bpfo = 0.5 * p['n'] * fr * (1 - (p['d'] / p['D']))
        bpfi = 0.5 * p['n'] * fr * (1 + (p['d'] / p['D']))
        bsf = 0.5 * (p['D'] / p['d']) * fr * (1 - (p['d'] / p['D'])**2)
        ftf = 0.5 * fr * (1 - (p['d'] / p['D']))
        theoretical_freqs[name] = {'BPFO': bpfo, 'BPFI': bpfi, 'BSF': bsf, 'FTF': ftf}

    # --- Part 1: 边带分析 (基于原始信号频谱) ---
    fft_vals = np.fft.fft(signal)
    fft_freq = np.fft.fftfreq(n_points, 1.0 / sampling_rate)
    positive_indices = np.where(fft_freq >= 0)
    freqs = fft_freq[positive_indices]
    amplitudes = np.abs(fft_vals[positive_indices])

    def get_energy_at(target_freq, freq_axis, amps, window=5):
        idx = np.argmin(np.abs(freq_axis - target_freq))
        start = max(0, idx - window)
        end = min(len(amps) - 1, idx + window)
        return np.sum(amps[start:end+1]**2)

    for name, freqs_dict in theoretical_freqs.items():
        # 内圈故障边带特征
        ir_carrier_e = get_energy_at(freqs_dict['BPFI'], freqs, amplitudes)
        ir_sideband_e = get_energy_at(freqs_dict['BPFI'] - fr, freqs, amplitudes) + \
                        get_energy_at(freqs_dict['BPFI'] + fr, freqs, amplitudes)
        adv_features[f'IR_Sideband_Energy_{name}'] = ir_sideband_e
        adv_features[f'IR_Sideband_Ratio_{name}'] = ir_sideband_e / (ir_carrier_e + 1e-9)
        
        # 滚动体故障边带特征
        b_carrier_e = get_energy_at(freqs_dict['BSF'], freqs, amplitudes)
        b_sideband_e = get_energy_at(freqs_dict['BSF'] - freqs_dict['FTF'], freqs, amplitudes) + \
                       get_energy_at(freqs_dict['BSF'] + freqs_dict['FTF'], freqs, amplitudes)
        adv_features[f'B_Sideband_Energy_{name}'] = b_sideband_e
        adv_features[f'B_Sideband_Ratio_{name}'] = b_sideband_e / (b_carrier_e + 1e-9)

    # --- Part 2: 包络解调分析 ---
    nyquist = 0.5 * sampling_rate
    low = 2000 / nyquist
    high = (nyquist - 10) / nyquist
    if low >= high: return adv_features

    b, a = butter(4, [low, high], btype='band')
    filtered_signal = filtfilt(b, a, signal)
    envelope = np.abs(hilbert(filtered_signal))
    
    env_fft_vals = np.fft.fft(envelope - np.mean(envelope))
    env_fft_freq = np.fft.fftfreq(len(envelope), 1.0 / sampling_rate)
    env_positive_indices = np.where(env_fft_freq >= 0)
    env_freqs = env_fft_freq[env_positive_indices]
    env_amplitudes = np.abs(env_fft_vals[env_positive_indices])
    
    def get_peak_at(target_freq, freq_axis, amps):
        idx = np.argmin(np.abs(freq_axis - target_freq))
        return amps[idx]

    for name, freqs_dict in theoretical_freqs.items():
        adv_features[f'Env_Peak_BPFO_{name}'] = get_peak_at(freqs_dict['BPFO'], env_freqs, env_amplitudes)
        adv_features[f'Env_Peak_BPFI_{name}'] = get_peak_at(freqs_dict['BPFI'], env_freqs, env_amplitudes)
        adv_features[f'Env_Peak_BSF_{name}'] = get_peak_at(freqs_dict['BSF'], env_freqs, env_amplitudes)

    return adv_features

def process_row_to_get_features(row: pd.Series, raw_data_root_dir: str, sampling_rate: int) -> pd.Series:
    """
    处理DataFrame中的单行数据，为其计算所有高级特征。
    """
    # 从行数据中重构.mat文件路径
    path_cols = [col for col in row.index if col.startswith('Level_') or col == 'OR_SubLevel']
    path_parts = [row[col] for col in path_cols if pd.notna(row[col])]
    mat_file_path = os.path.join(raw_data_root_dir, *path_parts)
    
    try:
        # 读取.mat文件并根据'Name'列获取当前行对应的信号序列
        data = scipy.io.loadmat(mat_file_path)
        raw_signal = data[row['Name']].flatten()
        
        # 获取RPM值并计算高级特征
        rpm = row['RPM']
        if pd.notna(rpm):
            advanced_features = calculate_advanced_features(raw_signal, rpm, sampling_rate)
            return pd.Series(advanced_features)
        else:
            # 如果行中没有RPM值，返回空Series
            return pd.Series({})
            
    except (FileNotFoundError, KeyError) as e:
        print(f"\n警告：无法处理行。文件 '{mat_file_path}' 或信号 '{row['Name']}' 未找到: {e}")
        return pd.Series({})
    except Exception as e:
        print(f"\n警告：处理行时发生未知错误: {e}")
        return pd.Series({})

# =============================================================================
# 2. 主执行函数
# =============================================================================
def main():
    """
    主程序：读取Excel，添加高级特征，保存为CSV。
    """
    # --- 配置区 ---
    # 1. 指定您提供的、包含基础特征的Excel文件路径
    input_excel_file = '../data/features/source/dataset.xlsx'
    
    # 2. 指定存放原始.mat文件的根文件夹路径
    raw_data_root_directory = '../data/raw/source' # '.' 代表此脚本所在的当前文件夹
    
    # 3. 设定采样率 (根据文档，源域数据通常为12kHz)
    SAMPLING_RATE = 12000
    # --- 配置结束 ---

    if not os.path.isfile(input_excel_file):
        print(f"错误：输入文件 '{input_excel_file}' 不存在。请检查文件名和路径。")
        return
    if not os.path.isdir(raw_data_root_directory):
        print(f"错误：原始数据根目录 '{raw_data_root_directory}' 不存在。请检查路径。")
        return

    try:
        df = pd.read_excel(input_excel_file)
    except Exception as e:
        print(f"读取Excel文件 '{input_excel_file}' 失败: {e}")
        print("请确保您已经安装了 'openpyxl' 库 (pip install openpyxl)")
        return
    
    print(f"成功读取 '{input_excel_file}'，共 {len(df)} 行数据。")
    print("开始计算高级特征，这将需要一些时间...")
    
    # 使用tqdm为apply方法添加进度条
    tqdm.pandas(desc="计算高级特征") 
    
    # 为每一行应用特征计算函数
    advanced_features_df = df.progress_apply(
        lambda row: process_row_to_get_features(row, raw_data_root_directory, SAMPLING_RATE), 
        axis=1
    )

    # 将新计算出的特征列合并回原始DataFrame
    final_df = pd.concat([df, advanced_features_df], axis=1)

    # 保存为CSV文件
    output_csv_file = '../data/features/source/dataset.csv'
    final_df.to_csv(output_csv_file, index=False)
    
    print(f"\n处理完成！")
    print(f"所有高级特征已添加，并成功保存到新文件: '{output_csv_file}'")
    print("\n最终输出数据预览 (仅显示部分新增列):")
    preview_cols = list(df.columns[:5]) + ['RPM', 'Name'] + list(advanced_features_df.columns[:4])
    print(final_df[preview_cols].head().to_string())


if __name__ == "__main__":
    main()

成功读取 'dataset.xlsx'，共 411 行数据。
开始计算高级特征，这将需要一些时间...


计算高级特征:   0%|          | 1/411 [00:00<00:00, 500.04it/s]


KeyError: 'Name'

In [12]:
import pandas as pd
import numpy as np
import scipy.io
from scipy.signal import hilbert, butter, filtfilt
import os
from tqdm import tqdm

# =============================================================================
# 1. 核心高级特征计算函数 (逻辑保持不变)
# =============================================================================
def calculate_advanced_features(signal: np.ndarray, rpm: float, sampling_rate: int) -> dict:
    """
    为单条信号计算所有高级特征（边带+包络），
    并且同时基于DE和FE两种轴承的参数进行计算。
    """
    n_points = len(signal)
    adv_features = {}

    fr = rpm / 60.0
    bearings = {
        'DE': {'n': 9, 'd': 0.3126, 'D': 1.537},
        'FE': {'n': 9, 'd': 0.2656, 'D': 1.122}
    }
    
    theoretical_freqs = {}
    for name, p in bearings.items():
        bpfo = 0.5 * p['n'] * fr * (1 - (p['d'] / p['D']))
        bpfi = 0.5 * p['n'] * fr * (1 + (p['d'] / p['D']))
        bsf = 0.5 * (p['D'] / p['d']) * fr * (1 - (p['d'] / p['D'])**2)
        ftf = 0.5 * fr * (1 - (p['d'] / p['D']))
        theoretical_freqs[name] = {'BPFO': bpfo, 'BPFI': bpfi, 'BSF': bsf, 'FTF': ftf}

    # Part 1: 边带分析
    fft_vals = np.fft.fft(signal)
    fft_freq = np.fft.fftfreq(n_points, 1.0 / sampling_rate)
    positive_indices = np.where(fft_freq >= 0)
    freqs = fft_freq[positive_indices]
    amplitudes = np.abs(fft_vals[positive_indices])

    def get_energy_at(target_freq, freq_axis, amps, window=5):
        idx = np.argmin(np.abs(freq_axis - target_freq))
        start = max(0, idx - window)
        end = min(len(amps) - 1, idx + window)
        return np.sum(amps[start:end+1]**2)

    for name, freqs_dict in theoretical_freqs.items():
        ir_carrier_e = get_energy_at(freqs_dict['BPFI'], freqs, amplitudes)
        ir_sideband_e = get_energy_at(freqs_dict['BPFI'] - fr, freqs, amplitudes) + \
                        get_energy_at(freqs_dict['BPFI'] + fr, freqs, amplitudes)
        adv_features[f'IR_Sideband_Energy_{name}'] = ir_sideband_e
        adv_features[f'IR_Sideband_Ratio_{name}'] = ir_sideband_e / (ir_carrier_e + 1e-9)
        
        b_carrier_e = get_energy_at(freqs_dict['BSF'], freqs, amplitudes)
        b_sideband_e = get_energy_at(freqs_dict['BSF'] - freqs_dict['FTF'], freqs, amplitudes) + \
                       get_energy_at(freqs_dict['BSF'] + freqs_dict['FTF'], freqs, amplitudes)
        adv_features[f'B_Sideband_Energy_{name}'] = b_sideband_e
        adv_features[f'B_Sideband_Ratio_{name}'] = b_sideband_e / (b_carrier_e + 1e-9)

    # Part 2: 包络解调分析
    nyquist = 0.5 * sampling_rate
    low = 2000 / nyquist
    high = (nyquist - 10) / nyquist
    if low >= high: return adv_features

    b, a = butter(4, [low, high], btype='band')
    filtered_signal = filtfilt(b, a, signal)
    envelope = np.abs(hilbert(filtered_signal))
    
    env_fft_vals = np.fft.fft(envelope - np.mean(envelope))
    env_fft_freq = np.fft.fftfreq(len(envelope), 1.0 / sampling_rate)
    env_positive_indices = np.where(env_fft_freq >= 0)
    env_freqs = env_fft_freq[env_positive_indices]
    env_amplitudes = np.abs(env_fft_vals[env_positive_indices])
    
    def get_peak_at(target_freq, freq_axis, amps):
        idx = np.argmin(np.abs(freq_axis - target_freq))
        return amps[idx]

    for name, freqs_dict in theoretical_freqs.items():
        adv_features[f'Env_Peak_BPFO_{name}'] = get_peak_at(freqs_dict['BPFO'], env_freqs, env_amplitudes)
        adv_features[f'Env_Peak_BPFI_{name}'] = get_peak_at(freqs_dict['BPFI'], env_freqs, env_amplitudes)
        adv_features[f'Env_Peak_BSF_{name}'] = get_peak_at(freqs_dict['BSF'], env_freqs, env_amplitudes)

    return adv_features

# =============================================================================
# 2. 主执行函数
# =============================================================================
def main():
    """
    主程序：读取CSV，独立遍历文件夹计算新特征，然后拼接。
    """
    # --- 配置区 ---
    input_csv_file = '../data/features/source/dataset.csv'
    raw_data_root_directory = '../data/raw/source' # '.' 代表此脚本所在的当前文件夹
    SAMPLING_RATE = 12000
    # --- 配置结束 ---

    # **步骤 1: 读取您已有的数据集**
    if not os.path.isfile(input_csv_file):
        print(f"错误：输入文件 '{input_csv_file}' 不存在。")
        return
    try:
        df_existing = pd.read_csv(input_csv_file)
        print(f"成功读取 '{input_csv_file}'，共 {len(df_existing)} 行数据。")
    except Exception as e:
        print(f"读取CSV文件 '{input_csv_file}' 失败: {e}")
        return

    # **步骤 2: 严格按升序遍历原始数据文件夹，生成信号列表**
    if not os.path.isdir(raw_data_root_directory):
        print(f"错误：原始数据根目录 '{raw_data_root_directory}' 不存在。")
        return
        
    signal_processing_order = []
    print("正在按升序遍历原始数据文件夹...")
    # os.walk默认就是自顶向下，我们需要对其中的文件夹和文件名进行排序
    for dirpath, dirnames, filenames in os.walk(raw_data_root_directory):
        # 按照文件夹名称升序遍历
        dirnames.sort()
        # 按照文件名称升序遍历
        for filename in sorted(filenames):
            if filename.endswith('.mat'):
                full_path = os.path.join(dirpath, filename)
                try:
                    data = scipy.io.loadmat(full_path)
                    # 确保mat文件内的信号键也按升序处理 (例如 DE, FE, BA)
                    signal_keys = sorted([key for key in data.keys() if key.endswith('_time')])
                    for key in signal_keys:
                        signal_processing_order.append({'path': full_path, 'key': key})
                except Exception as e:
                    print(f"读取或解析 {full_path} 时出错: {e}")

    # **步骤 3: 检查顺序和长度是否匹配**
    if len(df_existing) != len(signal_processing_order):
        print("\n警告！！！")
        print(f"您提供的CSV文件行数 ({len(df_existing)}) 与遍历文件夹找到的信号数量 ({len(signal_processing_order)}) 不匹配！")
        print("请检查您的数据和文件夹结构。尽管如此，程序将继续尝试按顺序计算。")

    # **步骤 4: 逐一计算高级特征**
    print("开始计算高级特征...")
    new_features_list = []
    # 使用RPM列作为一一对应的参数
    rpm_values = df_existing['rpm'].tolist()

    for i in tqdm(range(len(signal_processing_order)), desc="计算高级特征"):
        signal_info = signal_processing_order[i]
        rpm = rpm_values[i]
        
        try:
            data = scipy.io.loadmat(signal_info['path'])
            raw_signal = data[signal_info['key']].flatten()
            
            if pd.notna(rpm):
                advanced_features = calculate_advanced_features(raw_signal, rpm, SAMPLING_RATE)
                new_features_list.append(advanced_features)
            else:
                new_features_list.append({})
        except Exception as e:
            print(f"处理信号 {signal_info['key']} (来自 {signal_info['path']}) 时出错: {e}")
            new_features_list.append({})

    # **步骤 5: 合并新特征并保存**
    advanced_features_df = pd.DataFrame(new_features_list)
    final_df = pd.concat([df_existing, advanced_features_df], axis=1)

    output_csv_file = '../data/features/source/dataset_with_added_advanced_features.csv'
    final_df.to_csv(output_csv_file, index=False)

    print(f"\n处理完成！")
    print(f"所有高级特征已添加，并成功保存到新文件: '{output_csv_file}'")
    print("\n最终输出数据预览 (仅显示部分新增列):")
    preview_cols = list(df_existing.columns[:5]) + ['rpm', 'Name'] + list(advanced_features_df.columns[:4])
    print(final_df[preview_cols].head().to_string())

if __name__ == "__main__":
    main()

成功读取 'dataset.csv'，共 411 行数据。
正在按升序遍历原始数据文件夹...
开始计算高级特征...


计算高级特征: 100%|██████████| 411/411 [00:11<00:00, 34.52it/s]


处理完成！
所有高级特征已添加，并成功保存到新文件: 'dataset_with_added_advanced_features.csv'

最终输出数据预览 (仅显示部分新增列):


KeyError: "['Name'] not in index"

In [13]:
import pandas as pd
import numpy as np
import scipy.io
from scipy.signal import hilbert, butter, filtfilt
import os
from tqdm import tqdm

# =============================================================================
# 1. 核心高级特征计算函数 (逻辑保持不变)
# =============================================================================
def calculate_advanced_features(signal: np.ndarray, rpm: float, sampling_rate: int) -> dict:
    """
    为单条信号计算所有高级特征（边带+包络），
    并且同时基于DE和FE两种轴承的参数进行计算。
    """
    n_points = len(signal)
    adv_features = {}

    fr = rpm / 60.0
    bearings = {
        'DE': {'n': 9, 'd': 0.3126, 'D': 1.537},
        'FE': {'n': 9, 'd': 0.2656, 'D': 1.122}
    }
    
    theoretical_freqs = {}
    for name, p in bearings.items():
        bpfo = 0.5 * p['n'] * fr * (1 - (p['d'] / p['D']))
        bpfi = 0.5 * p['n'] * fr * (1 + (p['d'] / p['D']))
        bsf = 0.5 * (p['D'] / p['d']) * fr * (1 - (p['d'] / p['D'])**2)
        ftf = 0.5 * fr * (1 - (p['d'] / p['D']))
        theoretical_freqs[name] = {'BPFO': bpfo, 'BPFI': bpfi, 'BSF': bsf, 'FTF': ftf}

    # Part 1: 边带分析
    fft_vals = np.fft.fft(signal)
    fft_freq = np.fft.fftfreq(n_points, 1.0 / sampling_rate)
    positive_indices = np.where(fft_freq >= 0)
    freqs = fft_freq[positive_indices]
    amplitudes = np.abs(fft_vals[positive_indices])

    def get_energy_at(target_freq, freq_axis, amps, window=5):
        idx = np.argmin(np.abs(freq_axis - target_freq))
        start = max(0, idx - window)
        end = min(len(amps) - 1, idx + window)
        return np.sum(amps[start:end+1]**2)

    for name, freqs_dict in theoretical_freqs.items():
        ir_carrier_e = get_energy_at(freqs_dict['BPFI'], freqs, amplitudes)
        ir_sideband_e = get_energy_at(freqs_dict['BPFI'] - fr, freqs, amplitudes) + \
                        get_energy_at(freqs_dict['BPFI'] + fr, freqs, amplitudes)
        adv_features[f'IR_Sideband_Energy_{name}'] = ir_sideband_e
        adv_features[f'IR_Sideband_Ratio_{name}'] = ir_sideband_e / (ir_carrier_e + 1e-9)
        
        b_carrier_e = get_energy_at(freqs_dict['BSF'], freqs, amplitudes)
        b_sideband_e = get_energy_at(freqs_dict['BSF'] - freqs_dict['FTF'], freqs, amplitudes) + \
                       get_energy_at(freqs_dict['BSF'] + freqs_dict['FTF'], freqs, amplitudes)
        adv_features[f'B_Sideband_Energy_{name}'] = b_sideband_e
        adv_features[f'B_Sideband_Ratio_{name}'] = b_sideband_e / (b_carrier_e + 1e-9)

    # Part 2: 包络解调分析
    nyquist = 0.5 * sampling_rate
    low = 2000 / nyquist
    high = (nyquist - 10) / nyquist
    if low >= high: return adv_features

    b, a = butter(4, [low, high], btype='band')
    filtered_signal = filtfilt(b, a, signal)
    envelope = np.abs(hilbert(filtered_signal))
    
    env_fft_vals = np.fft.fft(envelope - np.mean(envelope))
    env_fft_freq = np.fft.fftfreq(len(envelope), 1.0 / sampling_rate)
    env_positive_indices = np.where(env_fft_freq >= 0)
    env_freqs = env_fft_freq[env_positive_indices]
    env_amplitudes = np.abs(env_fft_vals[env_positive_indices])
    
    def get_peak_at(target_freq, freq_axis, amps):
        idx = np.argmin(np.abs(freq_axis - target_freq))
        return amps[idx]

    for name, freqs_dict in theoretical_freqs.items():
        adv_features[f'Env_Peak_BPFO_{name}'] = get_peak_at(freqs_dict['BPFO'], env_freqs, env_amplitudes)
        adv_features[f'Env_Peak_BPFI_{name}'] = get_peak_at(freqs_dict['BPFI'], env_freqs, env_amplitudes)
        adv_features[f'Env_Peak_BSF_{name}'] = get_peak_at(freqs_dict['BSF'], env_freqs, env_amplitudes)

    return adv_features

# =============================================================================
# 2. 主执行函数
# =============================================================================
def main():
    """
    主程序：读取CSV，独立遍历文件夹计算新特征，然后拼接。
    """
    # --- 配置区 ---
    input_csv_file = '../data/features/source/dataset.csv'
    # **关键修正1：修正Windows路径的写法，使用正斜杠'/'或双反斜杠'\\'**
    raw_data_root_directory = '../data/raw/source'
    SAMPLING_RATE = 12000
    # --- 配置结束 ---

    # **步骤 1: 读取您已有的数据集**
    if not os.path.isfile(input_csv_file):
        print(f"错误：输入文件 '{input_csv_file}' 不存在。")
        return
    try:
        df_existing = pd.read_csv(input_csv_file)
        print(f"成功读取 '{input_csv_file}'，共 {len(df_existing)} 行数据。")
    except Exception as e:
        print(f"读取CSV文件 '{input_csv_file}' 失败: {e}")
        return

    # **步骤 2: 严格按升序遍历原始数据文件夹，生成信号列表**
    if not os.path.isdir(raw_data_root_directory):
        print(f"错误：原始数据根目录 '{raw_data_root_directory}' 不存在。")
        return
        
    signal_processing_order = []
    print("正在按升序遍历原始数据文件夹...")
    for dirpath, dirnames, filenames in os.walk(raw_data_root_directory):
        dirnames.sort()
        for filename in sorted(filenames):
            if filename.endswith('.mat'):
                full_path = os.path.join(dirpath, filename)
                try:
                    data = scipy.io.loadmat(full_path)
                    signal_keys = sorted([key for key in data.keys() if key.endswith('_time')])
                    for key in signal_keys:
                        signal_processing_order.append({'path': full_path, 'key': key})
                except Exception as e:
                    print(f"读取或解析 {full_path} 时出错: {e}")

    # **步骤 3: 检查顺序和长度是否匹配**
    if len(df_existing) != len(signal_processing_order):
        print("\n警告！！！")
        print(f"您提供的CSV文件行数 ({len(df_existing)}) 与遍历文件夹找到的信号数量 ({len(signal_processing_order)}) 不匹配！")
        print("请检查您的数据和文件夹结构。尽管如此，程序将继续尝试按顺序计算。")

    # **步骤 4: 逐一计算高级特征**
    print("开始计算高级特征...")
    
    # **关键修正2：智能查找RPM列，不再假设列名为'rpm'**
    rpm_col_name = None
    for col in df_existing.columns:
        if col.lower() == 'rpm':
            rpm_col_name = col
            print(f"已自动识别RPM列为: '{rpm_col_name}'")
            break
    
    if rpm_col_name is None:
        print("错误：在 '../data/features/source/dataset.csv' 中未找到RPM列。请确保文件中有一列的名称为'RPM'或'rpm'。")
        return
        
    rpm_values = df_existing[rpm_col_name].tolist()
    new_features_list = []

    for i in tqdm(range(len(signal_processing_order)), desc="计算高级特征"):
        signal_info = signal_processing_order[i]
        rpm = rpm_values[i]
        
        try:
            data = scipy.io.loadmat(signal_info['path'])
            raw_signal = data[signal_info['key']].flatten()
            
            if pd.notna(rpm):
                advanced_features = calculate_advanced_features(raw_signal, rpm, SAMPLING_RATE)
                new_features_list.append(advanced_features)
            else:
                new_features_list.append({})
        except Exception as e:
            print(f"处理信号 {signal_info['key']} (来自 {signal_info['path']}) 时出错: {e}")
            new_features_list.append({})

    # **步骤 5: 合并新特征并保存**
    advanced_features_df = pd.DataFrame(new_features_list)
    final_df = pd.concat([df_existing, advanced_features_df], axis=1)

    output_csv_file = '../data/features/source/dataset_with_added_advanced_features.csv'
    final_df.to_csv(output_csv_file, index=False)

    print(f"\n处理完成！")
    print(f"所有高级特征已添加，并成功保存到新文件: '{output_csv_file}'")
    
    # **关键修正3：使打印预览更健壮，只打印存在的列**
    print("\n最终输出数据预览 (仅显示部分新增列):")
    base_cols_to_preview = [col for col in df_existing.columns[:5] if col in final_df.columns]
    
    # 动态确定RPM和Name列是否存在于最终的DataFrame中
    rpm_and_name_cols = [col for col in [rpm_col_name, 'Name', 'name'] if col in final_df.columns]

    adv_cols_to_preview = []
    if not advanced_features_df.empty:
        adv_cols_to_preview = [col for col in advanced_features_df.columns[:4] if col in final_df.columns]
        
    preview_cols = base_cols_to_preview + rpm_and_name_cols + adv_cols_to_preview
    # 去重
    preview_cols = list(dict.fromkeys(preview_cols))
    
    print(final_df[preview_cols].head().to_string())

if __name__ == "__main__":
    main()

成功读取 'dataset.csv'，共 411 行数据。
正在按升序遍历原始数据文件夹...
开始计算高级特征...
已自动识别RPM列为: 'rpm'


计算高级特征: 100%|██████████| 411/411 [00:11<00:00, 34.81it/s]


处理完成！
所有高级特征已添加，并成功保存到新文件: 'dataset_with_added_advanced_features.csv'

最终输出数据预览 (仅显示部分新增列):
    Freq  PE DeLoc DeS  HP   rpm  IR_Sideband_Energy_DE  IR_Sideband_Ratio_DE  B_Sideband_Energy_DE  B_Sideband_Ratio_DE
0  12kHz  DE     B   7   0  1796           75945.759456              0.412436            101.178463            28.449328
1  12kHz  DE     B   7   0  1796           88528.848622              0.326829            220.682001           243.370334
2  12kHz  DE     B   7   0  1796          121968.822506              0.709392            193.059278            30.044882
3  12kHz  DE     B   7   1  1772           80183.384783              0.210407            105.865224           113.373285
4  12kHz  DE     B   7   1  1772          116916.027139              0.328002            215.434335            40.904997
